In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

print("All imports successful!")

All imports successful!


In [3]:
# Load the cleaned dataset
df = pd.read_csv(r'C:\Users\pc\rag-complaint-chatbot\data\processed\filtered_complaints.csv')

print(f"Full dataset shape: {df.shape}")
print(f"\nCategory distribution:")
print(df['product_category'].value_counts())

# Stratified sample - sample from each category separately
samples = []
categories = df['product_category'].unique()
per_category = 2500  # 4 categories x 2500 = 10000 total

for category in categories:
    cat_df = df[df['product_category'] == category]
    n = min(per_category, len(cat_df))
    samples.append(cat_df.sample(n, random_state=42))

df_sample = pd.concat(samples, ignore_index=True)

print(f"\nSample shape: {df_sample.shape}")
print(f"\nSample category distribution:")
print(df_sample['product_category'].value_counts())

Full dataset shape: (477183, 9)

Category distribution:
product_category
Credit Card        188779
Savings Account    154544
Money Transfer      98445
Personal Loan       35415
Name: count, dtype: int64

Sample shape: (10000, 9)

Sample category distribution:
product_category
Credit Card        2500
Savings Account    2500
Personal Loan      2500
Money Transfer     2500
Name: count, dtype: int64


In [4]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)

# Chunk all narratives and keep metadata
all_chunks = []

for idx, row in df_sample.iterrows():
    chunks = text_splitter.split_text(row['cleaned_narrative'])
    
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            'chunk_text': chunk,
            'complaint_id': row['complaint_id'],
            'product_category': row['product_category'],
            'issue': row['issue'],
            'company': row['company'],
            'state': row['state'],
            'chunk_index': i,
            'total_chunks': len(chunks)
        })

df_chunks = pd.DataFrame(all_chunks)

print(f"Total chunks created: {len(df_chunks)}")
print(f"\nChunks per category:")
print(df_chunks['product_category'].value_counts())
print(f"\nSample chunk:")
print(df_chunks['chunk_text'].iloc[0])

Total chunks created: 29024

Chunks per category:
product_category
Savings Account    7828
Credit Card        7606
Personal Loan      7293
Money Transfer     6297
Name: count, dtype: int64

Sample chunk:
the current issue is causing me significant distress and is having a negative impact on my sleep. it is challenging for me to accept this situation. i implore you to take immediate action to update my account.


In [5]:
# Load the embedding model
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded!")

# Get all chunk texts
texts = df_chunks['chunk_text'].tolist()

print(f"\nGenerating embeddings for {len(texts)} chunks...")
print("This will take a few minutes...")

# Generate embeddings in batches
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\nEmbeddings shape: {embeddings.shape}")
print("Embeddings generated successfully!")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded!

Generating embeddings for 29024 chunks...
This will take a few minutes...


Batches:   0%|          | 0/454 [00:00<?, ?it/s]


Embeddings shape: (29024, 384)
Embeddings generated successfully!


In [6]:
# Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype('float32'))

print(f"FAISS index created!")
print(f"Total vectors in index: {index.ntotal}")

# Save everything
os.makedirs(r'C:\Users\pc\rag-complaint-chatbot\vector_store', exist_ok=True)

# Save FAISS index
faiss.write_index(index, r'C:\Users\pc\rag-complaint-chatbot\vector_store\complaints.index')

# Save chunk metadata
df_chunks.to_csv(r'C:\Users\pc\rag-complaint-chatbot\vector_store\chunks_metadata.csv', index=False)

# Save embeddings
np.save(r'C:\Users\pc\rag-complaint-chatbot\vector_store\embeddings.npy', embeddings)

print(f"\nAll files saved to vector_store/")
print(f"- complaints.index")
print(f"- chunks_metadata.csv")
print(f"- embeddings.npy")

FAISS index created!
Total vectors in index: 29024

All files saved to vector_store/
- complaints.index
- chunks_metadata.csv
- embeddings.npy


In [7]:
# Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype('float32'))

print(f"FAISS index created!")
print(f"Total vectors in index: {index.ntotal}")

# Save everything
os.makedirs(r'C:\Users\pc\rag-complaint-chatbot\vector_store', exist_ok=True)

# Save FAISS index
faiss.write_index(index, r'C:\Users\pc\rag-complaint-chatbot\vector_store\complaints.index')

# Save chunk metadata
df_chunks.to_csv(r'C:\Users\pc\rag-complaint-chatbot\vector_store\chunks_metadata.csv', index=False)

# Save embeddings
np.save(r'C:\Users\pc\rag-complaint-chatbot\vector_store\embeddings.npy', embeddings)

print(f"\nAll files saved to vector_store/")
print(f"- complaints.index")
print(f"- chunks_metadata.csv")
print(f"- embeddings.npy")

FAISS index created!
Total vectors in index: 29024

All files saved to vector_store/
- complaints.index
- chunks_metadata.csv
- embeddings.npy
